In [1]:
from typing import List, Tuple
import asyncio
import time
import re


In [2]:
server_map: str = None
adapter_dirs = None
input_requests: List[Tuple[str, str, str, int, int]] = None
backend: str = "system"
output = None
debug=False,

In [3]:
# start = time.time()
# tasks: List[asyncio.Task] = []
# last_time = input_requests[0]
# step = 30
# for req in input_requests:
#     # print(req.req_id)
#     if req.req_time > last_time.req_time // 1 + step:
#         demand_tps = {a:0 for a in adapter_dirs}
#         index = input_requests.index(last_time)
#         while input_requests[index].req_time < req.req_time:
#             # rank = int(re.search(r'rank-(\d+)', input_requests[index].adapter_dir).group(1))
#             demand_tps[input_requests[index].adapter_dir] = demand_tps.get(input_requests[index].adapter_dir) + (input_requests[index].prompt_len + input_requests[index].output_len) / step
#             index += 1
#         # print(demand_tps)
#         adapter_demand = []
#         for adapter, tps in demand_tps.items():
#             rank = int(re.search(r'rank-(\d+)', adapter).group(1))
#             adapter_demand.append((rank, tps, adapter)) # tps here is expected tps
#         adapter_demand.sort(reverse=True)

#         # server_tps = {8: 2400, 16: 2100, 32: 1900, 64:1700, 128:1600} # operating point, fn of max rank
#         server_tps = {8: 2725, 16: 2700, 32: 2675, 64: 2625, 128: 2525} # operating point, fn of max rank

#         def is_compatible(group, tuple, scale=1):
#             ranks = [r for  r, _, _ in group]
#             ranks.append(tuple[0])
#             max_rank = max(ranks)
#             tps = sum([tps for _, tps, _ in group]) + tuple[1]
#             return tps <= (server_tps[max_rank] * scale)

#         def get_server_score_inclusive(group, tuple, server_tps):
#             ranks = [r for r, _, _ in group]
#             ranks.append(tuple[0])
#             max_rank = max(ranks)
#             tps = sum([tps for _, tps, _ in group]) + tuple[1]
#             return tps / server_tps[max_rank] if max_rank in server_tps else 0
        
#         def get_avg_server_score(idx, remaining_servers_count, adapter_demand, avg_server_tps):
#             if remaining_servers_count == 0 or len(adapter_demand) == 0 or len(adapter_demand) == idx + 1:
#                 return 0

#             remaining_tps = sum([tps for _, tps, _ in adapter_demand[idx + 1:]])
#             avg_server_score = remaining_tps / (remaining_servers_count * avg_server_tps)
#             return avg_server_score

#         servers = sorted(list(set(server_map.values())))
#         adapter_groups = [[] for _ in servers]
#         current_server = 0
#         num_servers = len(servers)
#         scores = [0] * num_servers
#         avg_server_tps = sum(server_tps.values()) / len(server_tps.keys())

#         for i, adapter_tuple in enumerate(adapter_demand):
#             if is_compatible(adapter_groups[current_server], adapter_tuple):
#                 new_server_score = get_server_score_inclusive(adapter_groups[current_server], adapter_tuple, server_tps)
#                 avg_server_score = get_avg_server_score(i + 1, num_servers - current_server - 1, adapter_demand, avg_server_tps)
#                 if avg_server_score == 0 or new_server_score <= avg_server_score:
#                     adapter_groups[current_server].append(adapter_tuple)
#                     scores[current_server] = new_server_score
#                 else:
#                     current_server += 1
#                     adapter_groups[current_server].append(adapter_tuple)
#                     scores[current_server] = get_server_score_inclusive(adapter_groups[current_server], adapter_tuple, server_tps)
#             elif current_server + 1 < num_servers:
#                 current_server += 1
#                 adapter_groups[current_server].append(adapter_tuple)
#                 scores[current_server] = get_server_score_inclusive(adapter_groups[current_server], adapter_tuple, server_tps)
#             else:
#                 with open("allocation_log.txt", "a") as f:
#                     f.write(f"{last_time} Overallocation for adapters from index {i} to {len(adapter_demand) - 1} ({(len(adapter_demand) - 1 - i + 1) / len(adapter_demand) * 100:.2f} % of total adapters)\n")
#                 for j in range(i, len(adapter_demand)):
#                     adapter_groups[j % num_servers].append(adapter_demand[j])
#                 break

#         print(adapter_groups)
#         with open("allocation_log.txt", "a") as f:
#             f.write(f"\n{last_time} Adapter groups:\n")
#             for i, group in enumerate(adapter_groups):
#                 f.write(f"  Server {servers[i]}: {[(adapter, tps) for _, tps, adapter in group]}\n")
#                 f.write(f"  Server {servers[i]} total tps: {sum([tps for _, tps, _ in group])}\n")
#             f.write(f"Scores: {scores}\n")

#         server_map = {}
#         for i, server in enumerate(servers):
#             for _, _, adapter in adapter_groups[i]:
#                 server_map[adapter] = server
#         print(server_map)
#         last_time = req
    

In [4]:
server_tps = {8: 2725, 16: 2700, 32: 2675, 64: 2625, 128: 2525} # operating point, fn of max rank

def is_compatible(group, tuple, scale=1):
    ranks = [r for  r, _, _ in group]
    ranks.append(tuple[0])
    max_rank = max(ranks)
    tps = sum([tps for _, tps, _ in group]) + tuple[1]
    return tps <= (server_tps[max_rank] * scale)

def get_server_score_inclusive(group, tuple, server_tps):
    ranks = [r for r, _, _ in group]
    ranks.append(tuple[0])
    max_rank = max(ranks)
    tps = sum([tps for _, tps, _ in group]) + tuple[1]
    return tps / server_tps[max_rank] if max_rank in server_tps else 0

def get_avg_server_score(idx, remaining_servers_count, adapter_demand, avg_server_tps):
    if remaining_servers_count == 0 or len(adapter_demand) == 0 or len(adapter_demand) == idx + 1:
        return 0

    remaining_tps = sum([tps for _, tps, _ in adapter_demand[idx + 1:]])
    avg_server_score = remaining_tps / (remaining_servers_count * avg_server_tps)
    return avg_server_score

servers = [1, 2]
adapter_groups = [[] for _ in servers]
current_server = 0
num_servers = len(servers)
scores = [0] * num_servers
avg_server_tps = sum(server_tps.values()) / len(server_tps.keys())
adapter_demand = [(128, 2000, 1), (128, 400, 2), (64, 2400, 1), (64, 200, 2)]

for i, adapter_tuple in enumerate(adapter_demand):
    if is_compatible(adapter_groups[current_server], adapter_tuple):
        new_server_score = get_server_score_inclusive(adapter_groups[current_server], adapter_tuple, server_tps)
        avg_server_score = get_avg_server_score(i + 1, num_servers - current_server - 1, adapter_demand, avg_server_tps)
        if avg_server_score == 0 or new_server_score <= avg_server_score:
            adapter_groups[current_server].append(adapter_tuple)
            scores[current_server] = new_server_score
        else:
            server_to_allocate = current_server
            if current_server + 1 < num_servers:
                server_to_allocate += 1
            adapter_groups[server_to_allocate].append(adapter_tuple)
            scores[server_to_allocate] = get_server_score_inclusive(adapter_groups[server_to_allocate], adapter_tuple, server_tps)
    elif current_server + 1 < num_servers:
        current_server += 1
        adapter_groups[current_server].append(adapter_tuple)
        scores[current_server] = get_server_score_inclusive(adapter_groups[current_server], adapter_tuple, server_tps)
    else:
        print(f"Overallocation for adapters from index {i} to {len(adapter_demand) - 1} ({(len(adapter_demand) - 1 - i + 1) / len(adapter_demand) * 100:.2f} % of total adapters)\n")
        for j in range(i, len(adapter_demand)):
            adapter_groups[j % num_servers].append(adapter_demand[j])
        break

    print(f"****Iteration {i}*****")
    print(adapter_groups)
    print("current server", current_server)
    print(scores)
    total_tps = {server: sum([tps for _, tps, _ in group]) for server, group in zip(servers, adapter_groups)}
    print(total_tps)


****Iteration 0*****
[[(128, 2000, 1)], []]
current server 0
[0.7920792079207921, 0]
{1: 2000, 2: 0}
****Iteration 1*****
[[(128, 2000, 1)], [(128, 400, 2)]]
current server 0
[0.7920792079207921, 0.31683168316831684]
{1: 2000, 2: 400}
****Iteration 2*****
[[(128, 2000, 1)], [(128, 400, 2), (64, 2400, 1)]]
current server 1
[0.7920792079207921, 2.0594059405940595]
{1: 2000, 2: 2800}
Overallocation for adapters from index 3 to 3 (25.00 % of total adapters)



In [5]:
server_tps = {8: 2725, 16: 2700, 32: 2675, 64: 2625, 128: 2525} # operating point, fn of max rank

def is_compatible(group, tuple, scale=1):
    ranks = [r for  r, _, _ in group]
    ranks.append(tuple[0])
    max_rank = max(ranks)
    tps = sum([tps for _, tps, _ in group]) + tuple[1]
    return tps <= (server_tps[max_rank] * scale)

def get_server_score(group, server_tps):
    ranks = [r for r, _, _ in group]
    max_rank = max(ranks)
    tps = sum([tps for _, tps, _ in group])
    return tps / server_tps[max_rank] if max_rank in server_tps else 0

def get_avg_server_score(idx, remaining_servers_count, adapter_demand, avg_server_tps):
    if remaining_servers_count == 0 or len(adapter_demand) == 0 or len(adapter_demand) == idx + 1:
        return 0

    remaining_tps = sum([tps for _, tps, _ in adapter_demand[idx + 1:]])
    avg_server_score = remaining_tps / (remaining_servers_count * avg_server_tps)
    return avg_server_score

servers = [1, 2]
adapter_groups = [[] for _ in servers]
current_server = 0
num_servers = len(servers)
scores = [0] * num_servers
avg_server_tps = sum(server_tps.values()) / len(server_tps.keys())
adapter_demand = [(128, 2000, 1), (128, 800, 2), (64, 1400, 1), (64, 100, 2)]
# adapter_demand = [(128, 2000, 1), (128, 400, 2), (128, 400, 2), (128, 400, 2), (64, 2400, 1), (64, 200, 2)]
# adapter_demand = [(128, 2000, 1), (128, 400, 2), (64, 2400, 1), (64, 200, 2)]

server_occupied_tps = [0] * num_servers
"""
for i, adapter_tuple in enumerate(adapter_demand):
    if is_compatible(adapter_groups[current_server], adapter_tuple):
        new_server_score = get_server_score_inclusive(adapter_groups[current_server], adapter_tuple, server_tps)
        avg_server_score = get_avg_server_score(i + 1, num_servers - current_server - 1, adapter_demand, avg_server_tps)
        if avg_server_score == 0 or new_server_score <= avg_server_score:
            adapter_groups[current_server].append(adapter_tuple)
            scores[current_server] = new_server_score
        else:
            server_to_allocate = current_server
            if current_server + 1 < num_servers:
                server_to_allocate += 1
            adapter_groups[server_to_allocate].append(adapter_tuple)
            scores[server_to_allocate] = get_server_score_inclusive(adapter_groups[server_to_allocate], adapter_tuple, server_tps)
    elif current_server + 1 < num_servers:
        current_server += 1
        adapter_groups[current_server].append(adapter_tuple)
        scores[current_server] = get_server_score_inclusive(adapter_groups[current_server], adapter_tuple, server_tps)
    else:
        print(f"Overallocation for adapters from index {i} to {len(adapter_demand) - 1} ({(len(adapter_demand) - 1 - i + 1) / len(adapter_demand) * 100:.2f} % of total adapters)\n")
        for j in range(i, len(adapter_demand)):
            adapter_groups[j % num_servers].append(adapter_demand[j])
        break
    """

for rank in sorted(server_tps.keys(), reverse=True):
    # get the adapters of rank rank from adapter_demand
    adapters = [x for x in adapter_demand if x[0] == rank]
    total_tps = sum([x[1] for x in adapters])
    if total_tps == 0: continue

    if total_tps + server_occupied_tps[current_server] <= server_tps[rank]:
        print(f"Rank {rank} adapters can be allocated to server {current_server} with total tps {total_tps}")
        adapter_groups[current_server].extend(adapters)
        server_occupied_tps[current_server] += total_tps
        scores[current_server] = get_server_score(adapter_groups[current_server], server_tps)
    else:
        # fit as many as possible in this server.
        for adapter in adapters:
            if is_compatible(adapter_groups[current_server], adapter):
                adapter_groups[current_server].append(adapter)
                server_occupied_tps[current_server] += adapter[1]
                scores[current_server] = get_server_score(adapter_groups[current_server], server_tps)
            else:
                # from here, out of room on the current server
                if current_server + 1 < num_servers:
                    current_server += 1
                else:
                    print("overallocation")
                adapter_groups[current_server].append(adapter)
                server_occupied_tps[current_server] += adapter[1]
                scores[current_server] = get_server_score(adapter_groups[current_server], server_tps)

print(adapter_groups)
print(scores)

Rank 64 adapters can be allocated to server 1 with total tps 1500
[[(128, 2000, 1)], [(128, 800, 2), (64, 1400, 1), (64, 100, 2)]]
[0.7920792079207921, 0.9108910891089109]


In [6]:
server_tps = {8: 2725, 16: 2700, 32: 2675, 64: 2625, 128: 2525} # operating point, fn of max rank

def is_compatible(group, tuple, scale=1):
    """
    Check if the given adapter tuple can fit within the group
    An adapter can fit if it is within the tps limit
    The tps limit depends on the max rank of the allocated adapters to this server
    """
    ranks = [r for  r, _, _ in group]
    ranks.append(tuple[0])
    max_rank = max(ranks)
    tps = sum([tps for _, tps, _ in group]) + tuple[1]
    return tps <= (server_tps[max_rank] * scale)

def get_server_score_inclusive(group, tuple, server_tps):
    """
    Get the score of the server (input tps / max supported tps based on max rank)
    including the new adapter tuple
    """
    ranks = [r for r, _, _ in group]
    ranks.append(tuple[0])
    max_rank = max(ranks)
    tps = sum([tps for _, tps, _ in group]) + tuple[1]
    return tps / server_tps[max_rank] if max_rank in server_tps else 0
    

def get_avg_server_score(remaining_servers_count: int, adapter_demand: list, adapters_placed: list[bool], avg_server_tps: float):
    """
    Get the average score of the future servers based on tps of unallocated adapters
    ! Note that this assumes the future servers are empty. Maybe we should account for their existing tps from the server_occupied_tps list
    """
    unplaced_adapters = [x for i, x in enumerate(adapter_demand) if not adapters_placed[i]]
    remaining_tps = sum([tps for _, tps, _ in unplaced_adapters])
    if remaining_servers_count == 0 or len(adapter_demand) == 0 or all(adapters_placed) or remaining_tps == 0:
        return 0

    avg_server_score = remaining_tps / (remaining_servers_count * avg_server_tps)
    return avg_server_score

servers = [1, 2]
adapter_groups = [[] for _ in servers]
current_server = 0
num_servers = len(servers)
scores = [0] * num_servers
avg_server_tps = sum(server_tps.values()) / len(server_tps.keys())
server_occupied_tps = [0] * num_servers

adapter_demand = [(128, 2000, 1), (128, 800, 2), (64, 1400, 1), (64, 100, 2)]
# adapter_demand = [(128, 2000, 1), (128, 400, 2), (128, 400, 2), (128, 400, 2), (64, 2400, 1), (64, 200, 2)]
adapter_demand = [(128, 2000, 1), (128, 400, 2), (64, 2400, 1), (64, 100, 2), (8, 100, 1)]
adapters_placed = [False] * len(adapter_demand)

greedy_overallocate: bool = True
for adapter_idx, adapter in enumerate(adapter_demand):
    for group_idx, group in enumerate(adapter_groups):
        if is_compatible(group, adapter):
            # check the impact on score of this server and assign
            new_server_score = get_server_score_inclusive(group, adapter, server_tps)
            avg_remaining_server_score = get_avg_server_score(num_servers - group_idx - 1, adapter_demand, adapters_placed, avg_server_tps)
            if avg_remaining_server_score == 0 or new_server_score <= avg_remaining_server_score:
                # allocate here
                adapter_groups[group_idx].append(adapter)
                server_occupied_tps[group_idx] += adapter[1]
                scores[group_idx] = new_server_score
                adapters_placed[adapter_idx] = True
                break

    if not adapters_placed[adapter_idx]:
        # * we did not find any server compatible with the adapter.
        # option 1: assign to the least full server now (greedy overallocate)
        # option 2: schedule everything else and then re-evaluate
        print("overallocation for adapter", adapter)
        if greedy_overallocate: # (option 1)
            min_server_idx = server_occupied_tps.index(min(server_occupied_tps))
            new_server_score = get_server_score_inclusive(adapter_groups[min_server_idx], adapter, server_tps)
            adapter_groups[min_server_idx].append(adapter)
            server_occupied_tps[min_server_idx] += adapter[1]
            scores[min_server_idx] = new_server_score
            adapters_placed[adapter_idx] = True

if not greedy_overallocate: # (option 2)
    for adapter_idx, adapter in enumerate(adapter_demand):
        if not adapters_placed[adapter_idx]:
            # TODO can change to first fit/best fit/score aware?
            min_server_idx = server_occupied_tps.index(min(server_occupied_tps))
            new_server_score = get_server_score_inclusive(adapter_groups[min_server_idx], adapter, server_tps)
            adapter_groups[min_server_idx].append(adapter)
            server_occupied_tps[min_server_idx] += adapter[1]
            scores[min_server_idx] = new_server_score
            adapters_placed[adapter_idx] = True
    

# print the total tps per server
print(server_occupied_tps)
print(adapter_groups)
print(scores)

[2400, 2600]
[[(128, 2000, 1), (128, 400, 2)], [(64, 2400, 1), (64, 100, 2), (8, 100, 1)]]
[0.9504950495049505, 0.9904761904761905]


---
# Non score based

In [10]:
from math import ceil, floor
from icecream import ic

server_tps = {8: 2725, 16: 2700, 32: 2675, 64: 2625, 128: 2525} # operating point, fn of max rank
servers = [1, 2]
adapter_demand = [(128, 2000, 1), (128, 800, 2), (64, 1400, 1), (64, 100, 2)]
adapter_demand = [(128, 2000, 1), (128, 400, 2), (128, 400, 2), (128, 400, 2), (64, 400, 1), (64, 200, 2)]
# adapter_demand = [(128, 2000, 1), (128, 400, 2), (64, 100, 2), (16, 1350, 1)]

def is_compatible(group, tuple, scale=1):
    """
    Check if the given adapter tuple can fit within the group
    An adapter can fit if it is within the tps limit
    The tps limit depends on the max rank of the allocated adapters to this server
    """
    ranks = [r for  r, _, _ in group]
    ranks.append(tuple[0])
    max_rank = max(ranks)
    tps = sum([tps for _, tps, _ in group]) + tuple[1]
    return tps <= (server_tps[max_rank] * scale)

adapter_groups = [[] for _ in servers]
num_servers = len(servers)
server_occupied_tps = [0] * num_servers
server_max_rank = [0] * num_servers
adapters_placed = [False] * len(adapter_demand)

# * checking compatibility
rank_instance_budget = [(rank, sum(tps for r, tps, _ in adapter_demand if r == rank) / rank_max_tps) for rank, rank_max_tps in server_tps.items()]
sorted_budgets = sorted(rank_instance_budget, key=lambda x: x[1], reverse=True)
print("sorted rank instance budgets:", sorted_budgets)
assert sum(budget for _, budget in rank_instance_budget) <= num_servers, "Exceeded server budget"

# * rounding
rounded_budgets = [(rank, round(budget)) for rank, budget in sorted_budgets]
print("rounded rank instance budgets:", rounded_budgets)
sum_rounded_off_budgets = sum(budget for _, budget in rounded_budgets)
if sum_rounded_off_budgets < num_servers:
    print("----Have instances left, moving to round up")
    idx = 0
    while sum_rounded_off_budgets < num_servers and idx < len(rounded_budgets):
        rounded_budgets[idx] = (rounded_budgets[idx][0], ceil(sorted_budgets[idx][1]))
        idx += 1
        sum_rounded_off_budgets = sum(budget for _, budget in rounded_budgets)

    print("rounded up:", rounded_budgets)
    
# * balanced allocation within assigned instances
adapters_with_assigned_instances = [x for x in rounded_budgets if x[1] > 0]
current_server = 0
for rank, num_assigned_instances in adapters_with_assigned_instances:
    l = current_server
    r = current_server + num_assigned_instances
    for adapter in adapter_demand:
        if adapter[0] == rank:
            least_occupied_server = min(range(l, r), key=lambda x: server_occupied_tps[x])
            if is_compatible(adapter_groups[least_occupied_server], adapter):
                adapter_groups[least_occupied_server].append(adapter)
                server_occupied_tps[least_occupied_server] += adapter[1]
                adapters_placed[adapter_demand.index(adapter)] = True
                server_max_rank[least_occupied_server] = max(server_max_rank[least_occupied_server], rank)
        
    current_server += num_assigned_instances

# * leftovers
for adapter_idx, adapter in enumerate(adapter_demand):
    if not adapters_placed[adapter_idx]:
        least_occupied_server = min(
            (i for i in range(num_servers) if server_max_rank[i] >= adapter[0]),
            key=lambda x: server_occupied_tps[x],
            default=None
        )
        if least_occupied_server is not None and is_compatible(adapter_groups[least_occupied_server], adapter):
            adapter_groups[least_occupied_server].append(adapter)
            server_occupied_tps[least_occupied_server] += adapter[1]
            adapters_placed[adapter_idx] = True

        if not adapters_placed[adapter_idx]:
            # we could not find a server with rank >= this adapters rank
            # need to colocate with a lower rank
            # TODO: better logic here - search through the closest ranks first and stop if we can fit
            new_least_occupied_server = min(range(num_servers), key=lambda x: server_occupied_tps[x])
            if is_compatible(adapter_groups[new_least_occupied_server], adapter):
                adapter_groups[new_least_occupied_server].append(adapter)
                server_occupied_tps[new_least_occupied_server] += adapter[1]
                adapters_placed[adapter_idx] = True

print("adapter_groups:", adapter_groups)
print("server_occupied_tps:", server_occupied_tps)
print("server max tps:", [server_tps[server_max_rank[i]] for i in range(num_servers)])
print("server max ranks:", server_max_rank)

sorted rank instance budgets: [(128, 1.2673267326732673), (64, 0.22857142857142856), (8, 0.0), (16, 0.0), (32, 0.0)]
rounded rank instance budgets: [(128, 1), (64, 0), (8, 0), (16, 0), (32, 0)]
----Have instances left, moving to round up
rounded up: [(128, 2), (64, 0), (8, 0), (16, 0), (32, 0)]
adapter_groups: [[(128, 2000, 1), (64, 400, 1)], [(128, 400, 2), (128, 400, 2), (128, 400, 2), (128, 400, 2), (128, 400, 2), (64, 200, 2)]]
server_occupied_tps: [2400, 2200]
server max tps: [2525, 2525]
server max ranks: [128, 128]


In [20]:
from math import ceil, floor
from icecream import ic
import heapq

server_tps = {8: 2725, 16: 2700, 32: 2675, 64: 2625, 128: 2525} # operating point, fn of max rank
servers = [1, 2]
adapter_demand = [(128, 2000, 1), (128, 800, 2), (64, 1400, 1), (64, 100, 2)]
adapter_demand = [(128, 2000, 1), (128, 400, 2), (128, 400, 3), (128, 400, 4), (64, 400, 1), (64, 200, 2)]
# adapter_demand = [(128, 2000, 1), (128, 400, 2), (64, 100, 2), (16, 1350, 1)]

def is_compatible(group_idx, tuple, scale=1):
    """
    Check if the given adapter tuple can fit within the group
    An adapter can fit if it is within the tps limit
    The tps limit depends on the max rank of the allocated adapters to this server
    """
    rank, tps, _ = tuple
    max_rank = max(rank, server_max_rank[group_idx])
    tps = server_occupied_tps[group_idx] + tuple[1]
    return tps <= (server_tps[max_rank] * scale)

adapter_groups = [[] for _ in servers]
num_servers = len(servers)
server_occupied_tps = [0] * num_servers
server_max_rank = [0] * num_servers
adapters_placed = [False] * len(adapter_demand)

# * checking compatibility
rank_instance_budget = [(rank, sum(tps for r, tps, _ in adapter_demand if r == rank) / rank_max_tps) for rank, rank_max_tps in server_tps.items()]
sorted_budgets = sorted(rank_instance_budget, key=lambda x: x[1], reverse=True)
print("sorted rank instance budgets:", sorted_budgets)
assert sum(budget for _, budget in rank_instance_budget) <= num_servers, "Exceeded server budget"

# * rounding
rounded_budgets = [(rank, round(budget)) for rank, budget in sorted_budgets]
print("rounded rank instance budgets:", rounded_budgets)
sum_rounded_off_budgets = sum(budget for _, budget in rounded_budgets)
if sum_rounded_off_budgets < num_servers:
    print("----Have instances left, moving to round up")
    idx = 0
    while sum_rounded_off_budgets < num_servers and idx < len(rounded_budgets):
        rounded_budgets[idx] = (rounded_budgets[idx][0], ceil(sorted_budgets[idx][1]))
        idx += 1
        sum_rounded_off_budgets = sum(budget for _, budget in rounded_budgets)

    print("rounded up:", rounded_budgets)
    
# * balanced allocation within assigned instances
adapters_with_assigned_instances = [x for x in rounded_budgets if x[1] > 0]
current_server = 0
for rank, num_assigned_instances in adapters_with_assigned_instances:
    l = current_server
    r = current_server + num_assigned_instances
    server_heap = [(server_occupied_tps[i], i) for i in range(l, r)]
    heapq.heapify(server_heap)
    for adapter_idx, adapter in enumerate(adapter_demand):
        if adapter[0] == rank:
            while server_heap:
                occupancy, least_occupied_server = heapq.heappop(server_heap)
                if is_compatible(least_occupied_server, adapter):
                    adapter_groups[least_occupied_server].append(adapter)
                    server_occupied_tps[least_occupied_server] += adapter[1]
                    adapters_placed[adapter_idx] = True
                    server_max_rank[least_occupied_server] = max(server_max_rank[least_occupied_server], rank)
                    heapq.heappush(server_heap, (server_occupied_tps[least_occupied_server], least_occupied_server))
                    break
        
    current_server += num_assigned_instances

# * leftovers
for adapter_idx, adapter in enumerate(adapter_demand):
    if not adapters_placed[adapter_idx]:
        least_occupied_server = min(
            (i for i in range(num_servers) if server_max_rank[i] >= adapter[0]),
            key=lambda x: server_occupied_tps[x],
            default=None
        )
        if least_occupied_server is not None and is_compatible(least_occupied_server, adapter):
            adapter_groups[least_occupied_server].append(adapter)
            server_occupied_tps[least_occupied_server] += adapter[1]
            adapters_placed[adapter_idx] = True
            continue

        # we could not find a server with rank >= this adapters rank
        # need to colocate with a lower rank
        # TODO: better logic here - search through the closest ranks first and stop if we can fit
        new_least_occupied_server = min(range(num_servers), key=lambda x: server_occupied_tps[x])
        if is_compatible(new_least_occupied_server, adapter):
            adapter_groups[new_least_occupied_server].append(adapter)
            server_occupied_tps[new_least_occupied_server] += adapter[1]
            adapters_placed[adapter_idx] = True
            server_max_rank[new_least_occupied_server] = max(server_max_rank[new_least_occupied_server], adapter[0])

print("adapter_groups:", adapter_groups)
print("server_occupied_tps:", server_occupied_tps)
print("server max tps:", [server_tps[server_max_rank[i]] for i in range(num_servers)])
print("server max ranks:", server_max_rank)

sorted rank instance budgets: [(128, 1.2673267326732673), (64, 0.22857142857142856), (8, 0.0), (16, 0.0), (32, 0.0)]
rounded rank instance budgets: [(128, 1), (64, 0), (8, 0), (16, 0), (32, 0)]
----Have instances left, moving to round up
rounded up: [(128, 2), (64, 0), (8, 0), (16, 0), (32, 0)]
adapter_groups: [[(128, 2000, 1)], [(128, 400, 2), (128, 400, 3), (128, 400, 4), (64, 400, 1), (64, 200, 2)]]
server_occupied_tps: [2000, 1800]
server max tps: [2525, 2525]
server max ranks: [128, 128]
